# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print('Available RecordSets:')

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"- {rs['@id']}: {rs.get('name', 'No name')} -- description: {rs.get('description', 'No description')}")
else:
    # Sometimes the 'record_sets' attribute may not exist or be empty, so check metadata.to_json()
    meta_json = metadata.to_json()
    record_sets = meta_json.get('recordSet', [])
    if not record_sets:
        print('No record sets found in the metadata.')
    else:
        for rs in record_sets:
            rid = rs.get('@id', 'Unknown id')
            name = rs.get('name', 'No name')
            desc = rs.get('description', 'No description')
            print(f"- {rid}: {name} -- description: {desc}")

### Inspecting a RecordSet's Fields
Let's list fields and their `@id` for the first available record set.

In [ ]:
# Try to discover record set IDs
meta_json = metadata.to_json()
record_sets = meta_json.get('recordSet', [])
if not record_sets:
    raise ValueError('No record sets found!')

# Pick the first one for demonstration
record_set_id = record_sets[0]['@id']

print(f"Inspecting RecordSet: {record_set_id}\n")
print('Fields in this RecordSet:')
fields = record_sets[0].get('field', [])
if isinstance(fields, dict):
    fields = [fields]
for f in fields:
    fid = f.get('@id', 'Unknown @id') if isinstance(f, dict) else str(f)
    fname = f.get('name', '') if isinstance(f, dict) else ''
    print(f"- {fid} {f'({fname})' if fname else ''}")

## 3. Data Extraction
Load data from the identified record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# List all available record set IDs
all_record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows for record set {rs_id}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Error loading record set {rs_id}: {e}")

# Show available DataFrame columns for the first record set with data
for rs_id, df in dataframes.items():
    print(f"\nColumns for record set {rs_id}:")
    print(df.columns.tolist())
    display(df.head())
    break  # Show only the first for brevity

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select record set and relevant field IDs
# We'll use the first loaded DataFrame (if any)
if not dataframes:
    raise ValueError('No dataframes loaded!')

active_record_set_id = list(dataframes.keys())[0]
df = dataframes[active_record_set_id].copy()

# Try to guess a numeric field (first floating point column)
import numpy as np
numeric_field = None
for col in df.columns:
    try:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    except Exception:
        # Try to coerce column to numeric
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field = col
                break
        except Exception:
            continue

if numeric_field is None:
    print('No numeric field automatically detected. Please examine df.columns and select one.')
else:
    print(f'Chosen numeric field: {numeric_field}')

if numeric_field is not None:
    # Drop NA values for correct filtering
    df = df[df[numeric_field].notna()]

    # Filter: show values > threshold
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by the first string field (if one exists)
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"\nGrouped data by '{group_field}', mean of {numeric_field}:")
        display(grouped_df.head())
    else:
        print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
if numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a group field exists, plot boxplot
    if group_field:
        # Truncate number of groups for plot clarity
        top_groups = df[group_field].value_counts().nlargest(10).index
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset offers mass spectrometry-based quantitative and descriptive information about extracellular vesicle proteins from human mast cells.
- Main available record sets and fields were auto-inspected; data extraction focused on the first available record set.
- Numeric fields were normalized and visualized, and data was optionally grouped by available categorical fields.
- For further domain-specific analysis, consult the Croissant metadata for explicit biological and measurement interpretations.